# Transformation Pipeline Validation & EDA

This notebook validates the data transformation pipeline and identifies areas for future transformations.

## Objectives
1. Validate all transformation functions work correctly
2. Analyze transformation outcomes and flag distributions
3. Identify data quality patterns requiring new transformations
4. Document findings for future pipeline improvements

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from data.transformations import (
    transform_dataframe,
    clean_first_name,
    clean_last_name,
    clean_middle_name,
    clean_suffix,
    clean_birth_date,
    clean_ssn,
    clean_email,
    clean_phone,
    clean_address,
    clean_city,
    clean_zip,
    clean_state,
    clean_sex_at_birth,
    process_name_redistribution,
    detect_camelcase,
    validate_name_presence
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', None)

---
## 1. Unit Tests for Individual Transformations

Test each transformation function with known inputs to validate correct behavior.

### 1.1 SSN Transformation Tests

In [ ]:
ssn_test_cases = [
    # (input, expected_output, expected_flags, description)
    ('123-45-6789', None, {'is_junk_SSN': True}, 'Sequential pattern should be junk'),
    ('123456789', None, {'is_junk_SSN': True}, 'Sequential without dashes'),
    ('987654321', None, {'is_junk_SSN': True}, 'Descending sequential'),
    ('000-12-3456', None, {'is_invalid_SSN': True}, 'Invalid area number 000'),
    ('666-12-3456', None, {'is_invalid_SSN': True}, 'Invalid area number 666'),
    ('900-12-3456', None, {'is_invalid_SSN': True}, 'ITIN range 9XX'),
    ('123-00-4567', None, {'is_invalid_SSN': True}, 'Invalid group number 00'),
    ('123-45-0000', None, {'is_invalid_SSN': True}, 'Invalid serial number 0000'),
    ('111111111', None, {'is_junk_SSN': True}, 'Repeating digits'),
    ('010101010', None, {'is_junk_SSN': True}, 'Known junk alternating'),
    ('111223333', None, {'is_junk_SSN': True}, 'Woolworth wallet card'),
    ('456-78-9012', '456789012', {}, 'Valid SSN'),
    ('', None, {'is_missing_SSN': True}, 'Empty string'),
    (None, None, {'is_missing_SSN': True}, 'None value'),
    ('1234567', '001234567', {'PADDED_SSN': True}, '7-digit padded'),
]

print("SSN Transformation Test Results:")
print("=" * 80)
all_passed = True
for inp, expected_out, expected_flags, desc in ssn_test_cases:
    result, flags = clean_ssn(inp)
    
    # Check output
    output_match = (pd.isna(result) and expected_out is None) or (result == expected_out)
    
    # Check flags
    flags_match = all(flags.get(k, False) == v for k, v in expected_flags.items())
    
    passed = output_match and flags_match
    status = "PASS" if passed else "FAIL"
    all_passed = all_passed and passed
    
    print(f"[{status}] {desc}")
    if not passed:
        print(f"       Input: {repr(inp)}")
        print(f"       Expected: {expected_out}, Got: {result}")
        print(f"       Expected flags: {expected_flags}, Got: {flags}")

print("=" * 80)
print(f"Overall: {'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED'}")

### 1.2 Email Transformation Tests

In [ ]:
email_test_cases = [
    # (input, expected_output, expected_flags, description)
    ('test@example.com', None, {'is_junk_Email': True}, 'Junk domain example.com'),
    ('noemail@gmail.com', None, {'is_junk_Email': True}, 'Junk prefix noemail'),
    ('unknown@unknown.com', None, {'is_junk_Email': True}, 'Known junk exact match'),
    ('ab@gmail.com', None, {'is_junk_Email': True}, 'Local part too short (2 chars)'),
    ('user123456@gmail.com', None, {'is_junk_Email': True}, 'Sequential digits in local'),
    ('valid.user@company.org', 'valid.user@company.org', {}, 'Valid email'),
    ('VALID@DOMAIN.COM', 'valid@domain.com', {}, 'Uppercase normalized'),
    ('missingat.com', None, {'is_invalid_Email': True}, 'Missing @ symbol'),
    ('nodomain@', None, {'is_invalid_Email': True}, 'Missing domain'),
    ('', None, {'is_missing_Email': True}, 'Empty string'),
    (None, None, {'is_missing_Email': True}, 'None value'),
    ('nan', None, {'is_junk_Email': True}, 'Primitive placeholder nan'),
]

print("Email Transformation Test Results:")
print("=" * 80)
all_passed = True
for inp, expected_out, expected_flags, desc in email_test_cases:
    result, flags = clean_email(inp)
    
    output_match = (pd.isna(result) and expected_out is None) or (result == expected_out)
    flags_match = all(flags.get(k, False) == v for k, v in expected_flags.items())
    
    passed = output_match and flags_match
    status = "PASS" if passed else "FAIL"
    all_passed = all_passed and passed
    
    print(f"[{status}] {desc}")
    if not passed:
        print(f"       Input: {repr(inp)}")
        print(f"       Expected: {expected_out}, Got: {result}")
        print(f"       Expected flags: {expected_flags}, Got relevant: {flags}")

print("=" * 80)
print(f"Overall: {'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED'}")

### 1.3 Name Redistribution Tests

In [ ]:
name_test_cases = [
    # (first, middle, last, suffix, expected_first, expected_middle, expected_last, description)
    ('JOHN MICHAEL SMITH', None, None, None, 'JOHN', 'MICHAEL', 'SMITH', '3-word first, empty last'),
    ('JOHN SMITH', None, None, None, 'JOHN', None, 'SMITH', '2-word first, empty last'),
    ('WILLIAM EARL', None, 'JOHNSON', None, 'WILLIAM', 'EARL', 'JOHNSON', '2-word first, last exists, move to middle'),
    ('MARY ANN', 'LOUISE', 'SMITH', None, 'MARY ANN', 'LOUISE', 'SMITH', '2-word first but middle exists, keep as-is'),
    ('JOHN DE LA CRUZ', None, None, None, 'JOHN', None, 'DE LA CRUZ', 'Compound last name prefix'),
    ('JOHN SMITH JR', None, None, None, 'JOHN', None, 'SMITH', 'Extract suffix during redistribution'),
]

print("Name Redistribution Test Results:")
print("=" * 80)
all_passed = True
for first, middle, last, suffix, exp_first, exp_middle, exp_last, desc in name_test_cases:
    res_first, res_middle, res_last, res_suffix, indicators = process_name_redistribution(first, middle, last, suffix)
    
    first_match = (pd.isna(res_first) and exp_first is None) or (res_first == exp_first)
    middle_match = (pd.isna(res_middle) and exp_middle is None) or (res_middle == exp_middle)
    last_match = (pd.isna(res_last) and exp_last is None) or (res_last == exp_last)
    
    passed = first_match and middle_match and last_match
    status = "PASS" if passed else "FAIL"
    all_passed = all_passed and passed
    
    print(f"[{status}] {desc}")
    if not passed:
        print(f"       Input: first={repr(first)}, middle={repr(middle)}, last={repr(last)}")
        print(f"       Expected: first={exp_first}, middle={exp_middle}, last={exp_last}")
        print(f"       Got: first={res_first}, middle={res_middle}, last={res_last}")
        print(f"       Indicators: {indicators}")

print("=" * 80)
print(f"Overall: {'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED'}")

### 1.4 CamelCase Detection Tests

In [ ]:
camelcase_test_cases = [
    ('JohnSmith', ['JOHN', 'SMITH'], 'Simple CamelCase'),
    ('WilliamEarl', ['WILLIAM', 'EARL'], 'Two names'),
    ('MariaDelCarmen', ['MARIA', 'DEL', 'CARMEN'], 'Three parts'),
    ('JOHN', [], 'All uppercase - no CamelCase'),
    ('john', [], 'All lowercase - no CamelCase'),
    ('John', [], 'Single word with capital'),
    ('JohnMichaelSmith', ['JOHN', 'MICHAEL', 'SMITH'], 'Three names'),
]

print("CamelCase Detection Test Results:")
print("=" * 80)
all_passed = True
for inp, expected, desc in camelcase_test_cases:
    result = detect_camelcase(inp)
    passed = result == expected
    status = "PASS" if passed else "FAIL"
    all_passed = all_passed and passed
    
    print(f"[{status}] {desc}")
    if not passed:
        print(f"       Input: {repr(inp)}")
        print(f"       Expected: {expected}, Got: {result}")

print("=" * 80)
print(f"Overall: {'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED'}")

### 1.5 Name Presence Validation Tests

In [ ]:
name_presence_test_cases = [
    ('JOHN', 'SMITH', True, {}, 'Both names present'),
    (None, 'SMITH', False, {'MISSING_FIRST_NAME': True}, 'Missing first name'),
    ('JOHN', None, False, {'MISSING_LAST_NAME': True}, 'Missing last name'),
    (None, None, False, {'MISSING_BOTH_NAMES': True}, 'Both names missing'),
    ('', 'SMITH', False, {'MISSING_FIRST_NAME': True}, 'Empty first name'),
    ('JOHN', '', False, {'MISSING_LAST_NAME': True}, 'Empty last name'),
]

print("Name Presence Validation Test Results:")
print("=" * 80)
all_passed = True
for first, last, exp_valid, exp_flags, desc in name_presence_test_cases:
    is_valid, flags = validate_name_presence(first, last)
    
    valid_match = is_valid == exp_valid
    flags_match = all(flags.get(k, False) == v for k, v in exp_flags.items())
    
    passed = valid_match and flags_match
    status = "PASS" if passed else "FAIL"
    all_passed = all_passed and passed
    
    print(f"[{status}] {desc}")
    if not passed:
        print(f"       Input: first={repr(first)}, last={repr(last)}")
        print(f"       Expected valid={exp_valid}, flags={exp_flags}")
        print(f"       Got valid={is_valid}, flags={flags}")

print("=" * 80)
print(f"Overall: {'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED'}")

---
## 2. Full Pipeline Validation with Sample Data

Create sample data that exercises all transformation paths.

In [ ]:
sample_data = pd.DataFrame({
    'PATID': range(1, 16),
    'FirstNM': [
        'John',           # Normal
        'MARY ANN',       # Compound first name with last
        'JohnSmith',      # CamelCase
        'JOHN MICHAEL SMITH',  # Full name in first
        'BABY BOY',       # Invalid pattern
        None,             # Missing
        'MR. ROBERT',     # Title to remove
        'José María',     # Unicode
        'WILLIAM EARL',   # Two words with existing last
        'DO NOT USE',     # Invalid
        'JOHN DE LA CRUZ',  # Compound last name prefix
        'SARAH',          # Normal
        'Test',           # Test record
        '',               # Empty
        'JOHN SMITH JR',  # Has suffix
    ],
    'MiddleNM': [
        'Michael', None, None, None, None, None, None, None, 
        None, None, None, 'Lynn', None, None, None
    ],
    'LastNM': [
        'Smith', 'Johnson', None, None, 'Doe', 'Wilson', 'Brown',
        'García', 'Williams', 'TEST', None, 'Davis', 'User', 'Blank', None
    ],
    'SuffixNM': [
        None, 'JR', None, None, None, '3RD', None, 
        None, 'SR', None, None, None, None, None, None
    ],
    'BirthDT': [
        '1990-05-15', '1985-12-01', '2000-01-01', '1900-01-01',
        '2025-06-01', '1880-01-01', '1975-03-20', '1992-08-10',
        '2030-01-01', '1988-11-05', '1995-07-22', '1960-02-28',
        None, 'invalid', '1982-09-14'
    ],
    'SSN': [
        '123-45-6789',   # Sequential junk
        '456-78-9012',   # Valid
        '111111111',     # Repeating junk
        '000-12-3456',   # Invalid area
        '666-12-3456',   # Invalid 666
        '900-12-3456',   # ITIN range
        '123-00-4567',   # Invalid group
        '123-45-0000',   # Invalid serial
        '234-56-7890',   # Valid
        None,            # Missing
        '',              # Empty
        '1234567',       # 7-digit to pad
        '010101010',     # Alternating junk
        '345678901',     # Valid
        '111223333',     # Woolworth
    ],
    'Email': [
        'john.smith@gmail.com',      # Valid
        'test@example.com',          # Junk domain
        'noemail@gmail.com',         # Junk prefix
        'ab@domain.com',             # Too short local
        'user123456@yahoo.com',      # Sequential digits
        'validuser@company.org',     # Valid
        'missingdomain@',            # Invalid
        None,                        # Missing
        '',                          # Empty
        'nan',                       # Primitive
        'UPPERCASE@DOMAIN.COM',      # Normalize
        'unknown@unknown.com',       # Known junk
        'real.person@real.org',      # Valid
        'nodot@domaincom',           # Invalid structure
        'good@email.net',            # Valid
    ],
    'PrimaryPhoneNBR': [
        '(312) 555-1234',  # Valid with formatting
        '1-800-555-9999',  # 11-digit with country code
        '1111111111',      # Repeating
        '312-555',         # Too short
        None,              # Missing
        '3125551234',      # Valid
        '000-555-1234',    # Invalid area code
        '1234567890',      # Sequential
        '773-555-4567',    # Valid
        '999-555-1234',    # Invalid area 999
        '',                # Empty
        '847.555.7890',    # Valid with dots
        '630 555 2345',    # Valid with spaces
        '12345',           # Too short
        '708-555-6789',    # Valid
    ],
    'AddressLine1': [
        '123 MAIN STREET',       # Full suffix
        '456 Oak Ave',           # Abbreviated
        'HOMELESS',              # Placeholder
        'P.O. Box 789',          # PO Box
        '100 NORTH MICHIGAN',    # Direction
        None,                    # Missing
        'UNKNOWN',               # Null value
        '200 Southwest Blvd',    # Direction
        'DO NOT USE',            # Invalid
        '300 First Street',      # Normal
        'Suite 100',             # Unit only
        '500 PARK AVENUE APT 5', # With unit
        '',                      # Empty
        '?',                     # Placeholder
        '600 Lake Shore Dr',     # Normal
    ],
    'CityNM': [
        'Chicago', 'CHCAGO', 'New York', 'Los Angeles', 'UNKNOWN',
        None, '', 'Houston', 'Phoenix', 'Philadelphia',
        'San Antonio', 'San Diego', 'Dallas', 'Austin', 'Jacksonville'
    ],
    'StateCD': [
        'IL', 'ILLINOIS', 'NY', 'CA', 'XX',
        None, '', 'TX', 'AZ', 'PA',
        'TX', 'CA', 'TX', 'TX', 'FL'
    ],
    'ZipCD': [
        '60601', '60602-1234', '00000', '99999', '123',
        None, '', '77001', '85001', '19101',
        '78201', '92101', '75201', '78701', '32099'
    ],
    'SexAtBirthDSC': [
        'M', 'FEMALE', 'Male', 'F', 'UNKNOWN',
        None, 'OTHER', 'MALE', 'Female', 'X',
        'M', 'F', '', 'Invalid', 'MALE'
    ]
})

print(f"Sample data created with {len(sample_data)} records")
sample_data.head()

In [ ]:
# Run full transformation pipeline
transformed_df = transform_dataframe(sample_data)

print(f"Transformation complete. Output has {len(transformed_df.columns)} columns.")
print(f"\nNew columns added:")
new_cols = [c for c in transformed_df.columns if c not in sample_data.columns]
for col in sorted(new_cols):
    print(f"  - {col}")

---
## 3. Transformation Outcomes Analysis

### 3.1 ValidRecord Analysis

In [ ]:
print("ValidRecord Distribution:")
print(transformed_df['ValidRecord'].value_counts())
print(f"\nInvalid records: {(~transformed_df['ValidRecord']).sum()} / {len(transformed_df)}")

invalid_records = transformed_df[~transformed_df['ValidRecord']]
if len(invalid_records) > 0:
    print("\nInvalid records details:")
    display_cols = ['PATID', 'FirstNM', 'LastNM', 'FirstNM_clean', 'LastNM_clean', 
                    'MISSING_FIRST_NAME', 'MISSING_LAST_NAME']
    display_cols = [c for c in display_cols if c in invalid_records.columns]
    print(invalid_records[display_cols].to_string())

### 3.2 Name Redistribution Analysis

In [ ]:
name_indicators = ['INFERRED_LAST_FROM_FIRST', 'MOVED_MIDDLE_FROM_FIRST', 
                   'CAMELCASE_SPLIT', 'EXTRACTED_SUFFIX', 'needs_name_review']
name_indicators = [c for c in name_indicators if c in transformed_df.columns]

print("Name Redistribution Flag Counts:")
for col in name_indicators:
    count = transformed_df[col].sum()
    print(f"  {col}: {count}")

# Show records that were redistributed
redistributed = transformed_df[transformed_df[name_indicators].any(axis=1)]
if len(redistributed) > 0:
    print(f"\n{len(redistributed)} records had name redistribution:")
    display_cols = ['PATID', 'FirstNM', 'MiddleNM', 'LastNM', 
                    'FirstNM_clean', 'MiddleNM_clean', 'LastNM_clean'] + name_indicators
    display_cols = [c for c in display_cols if c in redistributed.columns]
    print(redistributed[display_cols].to_string())

### 3.3 SSN Validation Analysis

In [ ]:
ssn_indicators = ['is_missing_SSN', 'is_junk_SSN', 'is_invalid_SSN', 'PADDED_SSN']
ssn_indicators = [c for c in ssn_indicators if c in transformed_df.columns]

print("SSN Validation Flag Counts:")
for col in ssn_indicators:
    count = transformed_df[col].sum()
    print(f"  {col}: {count}")

print("\nSSN Cleaning Results:")
ssn_cols = ['PATID', 'SSN', 'SSN_clean', 'SSN_Last4'] + ssn_indicators
ssn_cols = [c for c in ssn_cols if c in transformed_df.columns]
print(transformed_df[ssn_cols].to_string())

### 3.4 Email Validation Analysis

In [ ]:
email_indicators = ['is_missing_Email', 'is_junk_Email', 'is_invalid_Email']
email_indicators = [c for c in email_indicators if c in transformed_df.columns]

print("Email Validation Flag Counts:")
for col in email_indicators:
    count = transformed_df[col].sum()
    print(f"  {col}: {count}")

print("\nEmail Cleaning Results:")
email_cols = ['PATID', 'Email', 'Email_clean'] + email_indicators
email_cols = [c for c in email_cols if c in transformed_df.columns]
print(transformed_df[email_cols].to_string())

### 3.5 Quality Score Distribution

In [ ]:
if 'QUALITY_SCORE' in transformed_df.columns:
    print("Quality Score Statistics:")
    print(transformed_df['QUALITY_SCORE'].describe())
    
    print("\nQuality Tier Distribution:")
    quality_flags = ['LOW_QUALITY_RECORD', 'PARTIAL_IDENTITY_RECORD', 'HIGH_CONFIDENCE_DEMOGRAPHICS']
    for flag in quality_flags:
        if flag in transformed_df.columns:
            print(f"  {flag}: {transformed_df[flag].sum()}")

---
## 4. Load and Analyze Real Data (if available)

Load actual MDM_Population data for comprehensive analysis.

In [ ]:
# Check for real data files
import glob

data_files = glob.glob('../data/raw/*.csv') + glob.glob('../data/raw/*.parquet')
print("Available data files:")
for f in data_files:
    print(f"  - {f}")

if not data_files:
    print("  No data files found in ../data/raw/")
    print("  Place MDM_Population data file in data/raw/ to analyze real data.")

In [ ]:
# Uncomment and modify to load real data
# real_df = pd.read_csv('../data/raw/MDM_Population.csv')
# real_transformed = transform_dataframe(real_df)
# print(f"Loaded and transformed {len(real_transformed)} records")

---
## 5. Future Transformation Discovery

Analyze patterns in the data that may require new transformation rules.

### 5.1 Identify Frequent Values Needing Review

In [ ]:
def analyze_frequent_values(df, column, top_n=20):
    """Analyze most frequent values in a column for potential junk patterns."""
    if column not in df.columns:
        print(f"Column {column} not found")
        return None
    
    value_counts = df[column].value_counts().head(top_n)
    print(f"\nTop {top_n} values in {column}:")
    print(value_counts.to_string())
    
    # Flag potential issues
    suspicious = []
    for val, count in value_counts.items():
        if count > len(df) * 0.01:  # More than 1% of records
            suspicious.append((val, count))
    
    if suspicious:
        print(f"\n[!] Values appearing in >1% of records (potential junk):")
        for val, count in suspicious:
            print(f"    '{val}': {count} records ({count/len(df)*100:.1f}%)")
    
    return value_counts

# Analyze cleaned columns
analyze_frequent_values(transformed_df, 'Email_clean')

### 5.2 Pattern Analysis for New Junk Detection

In [ ]:
def find_suspicious_patterns(df, column, min_count=5):
    """
    Find suspicious patterns in values that passed cleaning.
    Returns values that appear frequently and may need new rules.
    """
    if column not in df.columns:
        return {}
    
    clean_col = f"{column}_clean" if f"{column}_clean" in df.columns else column
    
    # Get non-null values
    values = df[clean_col].dropna()
    
    # Find repeated values
    value_counts = values.value_counts()
    repeated = value_counts[value_counts >= min_count]
    
    return repeated

print("Checking for suspicious patterns that passed validation...")

# Check emails
suspicious_emails = find_suspicious_patterns(transformed_df, 'Email', min_count=2)
if len(suspicious_emails) > 0:
    print(f"\nEmails appearing multiple times (potential shared/fake):")
    print(suspicious_emails.to_string())

### 5.3 Identify Missing Transformation Categories

In [ ]:
def analyze_nullified_values(original_df, transformed_df, column):
    """
    Analyze what values were nullified during transformation.
    Helps identify patterns that might need explicit handling.
    """
    clean_col = f"{column}_clean" if f"{column}_clean" in transformed_df.columns else column
    
    if column not in original_df.columns or clean_col not in transformed_df.columns:
        return None
    
    # Find records where original had value but cleaned is null
    had_value = original_df[column].notna()
    now_null = transformed_df[clean_col].isna()
    nullified = had_value & now_null
    
    if nullified.sum() > 0:
        nullified_values = original_df.loc[nullified, column].value_counts()
        print(f"\n{column}: {nullified.sum()} values nullified during cleaning:")
        print(nullified_values.head(20).to_string())
    else:
        print(f"\n{column}: No values were nullified")
    
    return nullified

# Analyze nullified values for key fields
for col in ['SSN', 'Email', 'FirstNM', 'AddressLine1']:
    analyze_nullified_values(sample_data, transformed_df, col)

---
## 6. Summary & Recommendations

Document findings and suggest future improvements.

In [ ]:
print("=" * 80)
print("TRANSFORMATION PIPELINE VALIDATION SUMMARY")
print("=" * 80)

# Collect stats
stats = {
    'Total Records': len(transformed_df),
    'Valid Records': transformed_df['ValidRecord'].sum() if 'ValidRecord' in transformed_df.columns else 'N/A',
    'Invalid Records': (~transformed_df['ValidRecord']).sum() if 'ValidRecord' in transformed_df.columns else 'N/A',
}

# SSN stats
if 'SSN_clean' in transformed_df.columns:
    stats['Valid SSNs'] = transformed_df['SSN_clean'].notna().sum()
    stats['Junk SSNs'] = transformed_df['is_junk_SSN'].sum() if 'is_junk_SSN' in transformed_df.columns else 0
    stats['Invalid SSNs'] = transformed_df['is_invalid_SSN'].sum() if 'is_invalid_SSN' in transformed_df.columns else 0

# Email stats
if 'Email_clean' in transformed_df.columns:
    stats['Valid Emails'] = transformed_df['Email_clean'].notna().sum()
    stats['Junk Emails'] = transformed_df['is_junk_Email'].sum() if 'is_junk_Email' in transformed_df.columns else 0

# Name redistribution stats
if 'INFERRED_LAST_FROM_FIRST' in transformed_df.columns:
    stats['Names Redistributed'] = transformed_df['INFERRED_LAST_FROM_FIRST'].sum()
if 'CAMELCASE_SPLIT' in transformed_df.columns:
    stats['CamelCase Split'] = transformed_df['CAMELCASE_SPLIT'].sum()

print("\nKey Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

print("\n" + "=" * 80)
print("RECOMMENDATIONS FOR FUTURE TRANSFORMATIONS")
print("=" * 80)
print("""
1. REVIEW needed for records flagged with 'needs_name_review'
   - CamelCase splits may have edge cases
   - Long concatenated names may need manual review

2. CONSIDER adding nickname/diminutive mapping
   - BILL -> WILLIAM, BOB -> ROBERT, etc.
   - Would improve blocking scheme recall

3. MONITOR email frequency distribution post-cleaning
   - Any email appearing in 50+ records should be reviewed
   - May indicate clinic-default placeholder not yet in junk list

4. VALIDATE compound name prefix list completeness
   - Current list: DE LA, VAN, MC, etc.
   - May need expansion based on population demographics

5. ADD phone type classification
   - Mobile vs landline detection
   - May improve contact prioritization
""")

In [ ]:
# Final output summary
print("\nNotebook execution complete.")
print(f"Transformed DataFrame shape: {transformed_df.shape}")
print(f"Columns: {len(transformed_df.columns)}")